# AI as a Mathematical Collaborator — Weather modelling workshop

**UDSM MATH4SDG · Saturday 21 November 2026 · about 60 minutes**

**Goal:** Use generative AI to help estimate an unavailable temperature measurement, then investigate how trustworthy the proposed solution is.

We use **wholly synthetic daily observations** at two imaginary nearby weather stations. These are not real Tanzanian weather observations, nor data from TMA or any other partner. The dataset has no scientific conclusions about real climate.

You will use JupyterLab **and** an AI assistant. You can work alone or in pairs. Record your interaction with AI: what did you ask, what did it propose, and what did you verify?

**Confidentiality:** Do not paste restricted industrial information, personal records, or partner data into external AI services. This classroom dataset is safe to discuss.


## 1 · Understand the task (about 7 minutes)

**Scenario:** On some days, temperature at the *target station* is unavailable. We still know temperature at a nearby station, plus humidity and wind speed. Can we estimate the target temperature?

The table contains:

| Column | Meaning | Available when target temperature is missing? |
|---|---|---|
| `date` | Observation day | Yes |
| `nearby_temp_c` | Temperature at nearby station (°C) | Yes |
| `humidity_pct` | Relative humidity (%) | Yes |
| `wind_speed_m_s` | Wind speed (m/s) | Yes |
| `target_temp_c` | Temperature to estimate (°C) | **No** |

For model development, we use historical days on which the target station's temperature is known. We **pretend not to know** target temperatures when making predictions on later days. This is prediction from contemporaneous measurements, **not** a forecast of tomorrow's weather.

Open this notebook from `notebooks/` in the downloaded repository. Keep the `data/exercises/` folder in its original location.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

FEATURES = ["nearby_temp_c", "humidity_pct", "wind_speed_m_s"]
TARGET = "target_temp_c"

development = pd.read_csv(
    "../data/exercises/ai_weather_development.csv", parse_dates=["date"]
).sort_values("date").reset_index(drop=True)

# Use the first 210 days for training and the later days for validation.
train = development.iloc[:210].copy()
validation = development.iloc[210:].copy()

print("Total observations:", len(development))
print("Training days:", len(train), "| Validation days:", len(validation))
print("Training dates:", train["date"].min().date(), "to", train["date"].max().date())
print("Validation dates:", validation["date"].min().date(), "to", validation["date"].max().date())
display(development.head())


## 2 · Start with a simple baseline (about 5 minutes)

A baseline is the simplest sensible method to compare with. Our baseline predicts that the missing target temperature is **equal to the nearby-station temperature**.

For a set of predictions, we use the **mean absolute error** (MAE):

$$\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^n |y_i-\widehat y_i|.$$

Smaller is better. Since temperatures are measured in °C, the MAE is also in °C. Evaluate the baseline on the validation period.


In [ ]:
def baseline_predictor(train_rows, new_rows):
    # This baseline needs no fitting; 'train_rows' is unused.
    return new_rows["nearby_temp_c"].to_numpy()


def score(predictor, train_rows, evaluation_rows):
    # The predictor receives *no target temperatures* for evaluation days.
    available_inputs = evaluation_rows[FEATURES].copy()
    predictions = np.asarray(predictor(train_rows, available_inputs), dtype=float)
    if predictions.shape != (len(evaluation_rows),):
        raise ValueError("Return one temperature prediction per evaluation row.")
    if not np.isfinite(predictions).all():
        raise ValueError("Predictions contain missing or infinite values.")
    return mean_absolute_error(evaluation_rows[TARGET], predictions)

baseline_mae = score(baseline_predictor, train, validation)
print(f"Validation MAE — nearby-station baseline: {baseline_mae:.3f} °C")


**Discuss before asking AI:** What are the limitations of this baseline? Would a more complicated model necessarily be better?

## 3 · First AI experiment (about 10 minutes)

Open ChatGPT or another generative AI assistant. You may copy the following prompt, edit it, or write your own. **Do not paste the data-generating mechanism or any real partner data.**

> I am working in a Jupyter notebook using pandas, NumPy and scikit-learn. I have a DataFrame `train` containing columns `nearby_temp_c`, `humidity_pct`, `wind_speed_m_s` and `target_temp_c`. The target is `target_temp_c`; the other three columns are known when making a prediction. Please suggest a **simple predictive model** and write one Python function `predict_temperature(train_rows, new_rows)` that fits using `train_rows` and returns one predicted temperature per row of `new_rows`. The `new_rows` DataFrame contains **only the three input columns**, never the target. Do not access any validation or test labels; do not install packages. Briefly explain your choices.

Ask the AI for one code cell, and paste the **function definition** in the empty cell below. Run that cell. You can ask AI to debug errors, but record what changed.

**Important:** Never give a prediction function `validation[TARGET]` or `target_temp_c` for new days. The function must learn only from the supplied training rows.


In [ ]:
# Paste the AI-generated definition here, then run this cell.
# It should define: predict_temperature(train_rows, new_rows)
# Example structure (not a solution):
# def predict_temperature(train_rows, new_rows):
#     ...
#     return predictions


Run the following cell after your function is defined. It checks the output format and measures validation error. If the cell reports that the function is missing, return to the previous cell.


In [ ]:
if "predict_temperature" not in globals():
    print("Define predict_temperature(train_rows, new_rows) in the cell above first.")
else:
    first_model_mae = score(predict_temperature, train, validation)
    first_predict_temperature = predict_temperature  # Keep the first version.
    print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
    print(f"First AI-model validation MAE: {first_model_mae:.3f} °C")


**Your experiment log — edit this markdown cell:**

- My original request to AI: …
- Model or algorithm proposed: …
- Did the code run immediately? If not, what needed fixing? …
- What did the model assume? …
- Validation MAE, and comparison with the baseline: …

## 4 · Critique and revise (about 12 minutes)

Now ask your AI assistant to **criticize its own solution**. For example:

> Our baseline uses the nearby-station temperature. We used an earlier period for training and a later period for validation. Please inspect the modelling assumptions and potential data leakage in your solution. Suggest **one specific, testable improvement**. Explain why it may help and why it might fail. Give me a revised function named `revised_predict_temperature(train_rows, new_rows)` using the same interface and no new dependencies.

You can instead ask for a *simpler* model, or investigate whether residual errors depend on humidity. Do not tune repeatedly on the final test data, which you have not opened.

Paste the revised function below. **Keep the first function unchanged** so the two approaches can be compared.


In [ ]:
# Paste the revised function here, then run this cell.
# It should define: revised_predict_temperature(train_rows, new_rows)


In [ ]:
if "revised_predict_temperature" not in globals():
    print("Define revised_predict_temperature in the cell above first.")
else:
    revised_model_mae = score(revised_predict_temperature, train, validation)
    print(f"Baseline validation MAE: {baseline_mae:.3f} °C")
    if "first_model_mae" in globals():
        print(f"First AI-model validation MAE: {first_model_mae:.3f} °C")
    print(f"Revised AI-model validation MAE: {revised_model_mae:.3f} °C")


**Your experiment log — edit this markdown cell:**

- What did the AI criticize? …
- What did you change in your instructions? …
- What changed in the code or mathematical model? …
- Which claim about the revised model have you **checked independently**? …
- Is the revision more useful, and why? …

## 5 · Lock your choice *before* final testing (about 2 minutes)

Select the candidate with the smallest **validation MAE** (including the baseline). After selecting it, do not change the selected method based on the final test result. A more elaborate model is **not** automatically the best choice.

If you did not finish one of the AI models, you may still select the baseline. You can discuss with your facilitator.


In [ ]:
candidates = {"Nearby-station baseline": (baseline_mae, baseline_predictor)}
if "first_model_mae" in globals():
    candidates["First AI model"] = (first_model_mae, first_predict_temperature)
if "revised_model_mae" in globals():
    candidates["Revised AI model"] = (revised_model_mae, revised_predict_temperature)

chosen_name = min(candidates, key=lambda name: candidates[name][0])
chosen_validation_mae, chosen_predictor = candidates[chosen_name]
print(f"LOCKED CHOICE: {chosen_name}")
print(f"Validation MAE: {chosen_validation_mae:.3f} °C")
print("Do not change this choice after opening the final-test file.")


## 6 · Independent final test (about 7 minutes)

**STOP:** Open the final-test data only after you have chosen your model above. When ready, change `READY_FOR_FINAL_TEST = False` to `True` in the next code cell. This prevents `Run All` from revealing the answer prematurely. The final data are a *later time interval*, kept separate during model development. We fit the already-selected method again on all development observations, then evaluate it once on the final period.

The file contains the target temperatures solely so that this notebook can calculate error; the model function itself receives **only input features** for those days.


In [ ]:
READY_FOR_FINAL_TEST = False  # Change to True only after locking the choice.
if READY_FOR_FINAL_TEST:
    final_test = pd.read_csv(
        "../data/exercises/ai_weather_final.csv", parse_dates=["date"]
    ).sort_values("date").reset_index(drop=True)

    final_baseline_mae = score(baseline_predictor, development, final_test)
    final_chosen_mae = score(chosen_predictor, development, final_test)
    print(f"Final-test MAE — nearby-station baseline: {final_baseline_mae:.3f} °C")
    print(f"Final-test MAE — locked model ({chosen_name}): {final_chosen_mae:.3f} °C")
    print("Lower is better. Report the difference, even if the baseline wins.")
else:
    print('Final test locked. Change READY_FOR_FINAL_TEST to True when ready.')


## 7 · Group discussion (final 10 minutes)

**Each pair should report:**

1. What was one specific AI suggestion you accepted, revised or rejected?
2. Which model did you select *before* testing, and what was its final-test MAE?
3. Give one independently checked mathematical or computational claim.
4. What could go wrong when applying this method to **real** incomplete weather records (for instance, missing observations during extreme conditions)?
5. How would you adapt the AI-assisted workflow to your actual industrial project?

**Scientific cautions:** A low error on synthetic data does not prove an approach will work on real weather. The evaluation assumes we can observe humidity, wind and nearby-station temperature when the target station is missing. If missingness follows a non-random pattern, our test might be optimistic. This exercise demonstrates a modelling process, **not** an operational weather-imputation system.

**Takeaway:** Ask AI to propose and explain. Use mathematics, data and independently run tests to decide what to trust.
